# Fine-tune infgrad/Prism-Qwen3.5-Reranker-2B — ứng viên thay kênh `jina`

Model bên thứ 3 (infgrad), backbone "Qwen3.5" -- prompt/công thức điểm đã xác nhận qua trang HuggingFace của model (system prompt hơi khác bản Qwen gốc, không có câu "answer can only be yes or no"; điểm = sigmoid(l_yes-l_no), toán học giống hệt cách đọc của Qwen3-Reranker). Model có thể sinh thêm `<contribution>`/`<evidence>` sau "yes" nhưng notebook chỉ đọc logit ở đúng 1 vị trí đầu tiên (không gọi `.generate()`), nên phần sinh thêm đó không ảnh hưởng gì tới điểm số hay tốc độ.

Kiến trúc: **generative yes/no cross-encoder** (không phải sequence-classification
1-logit như jina gốc) — chấm điểm bằng `logit("yes") - logit("no")` ở vị trí
token kế tiếp sau prompt chat, đúng công thức cả hai model card của họ Qwen3
tài liệu (`sigmoid(l_yes - l_no)`). Xem `torch_common.causal_yesno_logits` /
`finetune_prism_reranker` để biết chi tiết.

**Vì sao chỉ thế được `jina`, không thế được `dense`/aiteamvn**: kênh `dense`
còn nuôi 2 tầng sinh ứng viên khác (dense expansion union-50, corpus dense
index) — cả hai cần **vector embedding** để tính similarity toàn corpus. Một
reranker generative yes/no chỉ chấm được từng cặp (câu hỏi, ứng viên) đã có
sẵn, không sinh được vector đó. Nên notebook này (và notebook Qwen3-Reranker
song song) chỉ có thể là ứng viên thay **kênh `jina`** trong LTR fusion —
`score_overrides={"jina": ...}`, đúng cơ chế `finetune_jina.py` đã dùng.
Output ghi vào thư mục **riêng** `prism_reranker/` (không đè lên `jina/`) để so sánh
model nào thắng khi cùng đóng vai `jina`.

**2B tham số, base "Qwen3.5" (bên thứ 3, infgrad)** — dùng **LoRA** (`peft`, rank=16) trên base đóng
băng fp16, KHÔNG full fine-tune (base infgrad/Prism-Qwen3.5-Reranker-2B quá lớn để full fine-tune
vừa VRAM Colab). Batch mặc định đã hạ tương ứng: `BATCH_SIZE=4`,
`ACCUM=4` (effective batch = 16),
`EVAL_BATCH_SIZE=16`, `MAX_LENGTH=1024`.

---

## Cách chạy để không mất kết quả

Notebook cắt thành từng cell nhỏ, **mỗi epoch một cell**. `WORK` nằm thẳng
trên Google Drive (`/content/drive/MyDrive/fine_tune_work/prism_reranker/`), nên kết
quả **tự động persist** sau mỗi epoch — không cần tải zip thủ công. Nếu
runtime Colab bị ngắt, cứ **chạy lại từ cell 1**: notebook tự đọc
`history.json` cũ, bỏ qua epoch đã ghi và nạp lại adapter LoRA tốt nhất
(`best_state.pt` — chỉ vài MB, không phải base model), không tốn lại GPU
của phần đã xong. Muốn làm lại sạch thì đặt `RESUME = False`.

**Runtime cần chọn: GPU (A100 nếu có; T4/L4 vẫn chạy được nhưng chậm hơn và
cần hạ `BATCH_SIZE`/`MAX_LENGTH` nếu OOM), high-RAM nếu có.**

## Cell 1 — Cài đặt thư viện

Colab thường có sẵn `transformers` nhưng bản cũ hơn mức Qwen3 (kiến trúc
dùng cho cả LLM và embedding/reranker) yêu cầu, và `peft` có thể chưa cài.
Chạy cell này **trước khi import bất cứ thứ gì khác** — nâng cấp sau khi
`torch`/`transformers` đã import vào runtime có thể không có tác dụng cho
tới khi restart.

In [ ]:
!pip install -q -U "transformers>=4.51.0" "peft>=0.10.0" accelerate

## Cell 2 — Cấu hình

`REF` mount qua Google Drive tại `/content/drive/MyDrive/ref` — bạn upload
**nguyên folder `ref/`** (results/ cache + DSC2026-LegalIR-main/ corpus +
models/ + code `fine_tune/`, giờ cần thêm 2 file `torch_common.py` đã cập
nhật và `finetune_prism_reranker.py`) lên đúng đường dẫn này trong Drive, giữ nguyên cấu
trúc con. `WORK` (checkpoint + log) ghi ra một thư mục Drive khác
(`fine_tune_work/`), tách khỏi `ref/` để không lẫn output vào input.

In [ ]:
import os, sys, json, time
from pathlib import Path

# Giảm phân mảnh bộ nhớ GPU — chính thông báo OOM của PyTorch gợi ý.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from google.colab import drive
drive.mount("/content/drive")

REF  = Path("/content/drive/MyDrive/ref")             # bạn upload nguyên folder ref/ vào đây
CODE = REF / "fine_tune"
WORK = Path("/content/drive/MyDrive/fine_tune_work")  # riêng khỏi ref/, tự persist qua session

EPOCHS          = 3
RESUME          = True     # False = bỏ qua history/checkpoint cũ, chạy lại từ đầu
LR              = 2e-4     # LoRA cần LR cao hơn full fine-tune (update qua ma trận rank-thấp
                            # có biên độ nhỏ hơn nhiều) -- không dùng 1e-5 như jina/aiteamvn
BATCH_SIZE      = 4
ACCUM           = 4          # effective batch = BATCH_SIZE*ACCUM = 16
NEGATIVES       = 4          # hard negative mỗi truy vấn mỗi bước
NEGATIVE_DEPTH  = 48       # lấy negative sâu bao nhiêu trong pool lexical đã cache
MAX_LENGTH      = 1024
EVAL_BATCH_SIZE = 16
MAX_GPUS        = 1        # Colab thường cấp 1 GPU -- không cần DataParallel
PASSAGES_PER_DOC_TRAIN = 1   # lúc train; lúc đánh giá luôn là 2 như pipeline gốc
TRAIN_QUERIES   = None     # None = dùng cả ~1650 truy vấn có cache (gồm cả 600 LOBO)
SEED            = 2026
LOSS            = "pairwise"
INSTRUCTION     = ('Given a Vietnamese legal question, determine whether the Document '
                   'contains the answer to the Query')

# --- LoRA ---
LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = "all-linear"

assert REF.exists(), f"Không thấy {REF} — mount Drive xong chưa? Đã upload folder ref/ đúng chỗ chưa?"
assert (CODE / "burst_common.py").exists(), f"Không thấy code trong {CODE}"
assert (CODE / "finetune_prism_reranker.py").exists(), (
    f"Không thấy finetune_prism_reranker.py trong {CODE} — đã upload bản code mới nhất "
    f"(kèm torch_common.py đã có causal_yesno_logits) chưa?")
sys.path.insert(0, str(CODE))
os.environ["BURST_ROOT"] = str(REF)
os.environ["BURST_WORK"] = str(WORK)
WORK.mkdir(parents=True, exist_ok=True)
print("dữ liệu:", REF)
print("code   :", CODE)
print("output :", WORK)

import torch
print("torch", torch.__version__, "| CUDA:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU rồi chạy lại"

import burst_common as bc
import torch_common as tc
import finetune_prism_reranker as ft
bc.check_code_version()
assert hasattr(tc, "causal_yesno_logits"), (
    "torch_common.py trong Drive chưa có causal_yesno_logits -- upload bản mới nhất.")

TAG = ft.TAG
print(f"model  : {ft.HF_REPO}")
print(f"tag    : {TAG}  (ghi vào {WORK / TAG}, thay thế kênh 'jina' khi chấm điểm)")

## Cell 3 — Dựng lại tập đánh giá 600 truy vấn từ cache

Chỉ CPU, khoảng 1–3 phút. Cell này chạy trước phần GPU có chủ đích: nếu
đường dẫn hay cache có vấn đề thì hỏng ở đây, trước khi tốn thời gian GPU.

Con số baseline in ra phải đúng bằng **recall 0.9511 / F2 0.6144** — đó là
bản đã nộp, và là baseline chung mọi kênh (kể cả `jina` gốc) so sánh vào.

In [ ]:
documents = bc.DocumentStore(REF / bc.DATA_SUBDIR, preload=True)
bundle    = bc.build_eval_bundle(REF, documents)

baseline, _, _ = bc.lobo_evaluate(bundle)
print(f"\nBASELINE (toàn bộ 6 kênh từ cache)")
print(f"  recall    = {baseline['recall']:.4f}   <- bản đã nộp: 0.9511")
print(f"  precision = {baseline['precision']:.4f}")
print(f"  f2        = {baseline['f2']:.4f}   <- bản đã nộp: 0.6144")
for name, b in baseline["blocks"].items():
    print(f"  block {name}: recall={b['recall']:.4f} f2={b['f2']:.4f}")

## Cell 4 — Tập huấn luyện

Toàn bộ truy vấn có nhãn **đã có sẵn cache retrieval** (~1.650 truy vấn,
index 0-1149 và 1250-1749) — bao gồm cả 600 truy vấn của block LOBO dùng để
đánh giá bên dưới. Hard negative lấy thẳng từ pool lexical đã cache — không
chạy lại retrieval.

⚠️ Vì tập train giờ **giao hoàn toàn** với tập đánh giá, Recall/F2 in ra sau
mỗi epoch không còn là ước lượng generalization — model đã thấy chính các
truy vấn đó lúc train. Đây là đánh đổi có chủ đích để tận dụng hết dữ liệu
nhãn, không phải một lỗi (xem README.md#dữ-liệu-train).

`RunRecorder` chọn `best_state.pt` theo **epoch tốt nhất của chính lần chạy
này** trên holdout 600 truy vấn: epoch đầu luôn được lưu, epoch sau chỉ ghi
đè khi validate tăng. Việc có vượt baseline cache hay không là một câu hỏi
khác, trả lời riêng ở cột `beats_baseline` — không quyết định chuyện lưu.

In [ ]:
examples = bc.build_train_pool(REF, negatives=NEGATIVE_DEPTH,
                               limit=TRAIN_QUERIES, seed=SEED,
                               documents=documents)

recorder = bc.RunRecorder(WORK, TAG, baseline, resume=RESUME)
print("Ghi kết quả vào:", recorder.dir)

## Cell 5 — Nạp model

Tải trọng số fp16 từ HuggingFace (infgrad/Prism-Qwen3.5-Reranker-2B), đóng băng, gắn LoRA r=16
lên mọi Linear layer, upcast riêng adapter LoRA lên fp32. Nếu đang resume,
adapter tốt nhất của lần chạy trước sẽ được nạp đè (nạp base fresh từ hub +
đè adapter, không lưu lại base — checkpoint chỉ vài MB).

Colab thường cấp **1 GPU**, nên `DataParallel` không có gì để chia — no-op.

In [ ]:
import torch

device, _, n_gpus = tc.setup(SEED)
if MAX_GPUS:
    n_gpus = min(n_gpus, MAX_GPUS)

model, tokenizer = ft.load_model(REF, ft.DEFAULT_MODEL, gradient_checkpointing=True,
                                 lora_r=LORA_R, lora_alpha=LORA_ALPHA,
                                 lora_dropout=LORA_DROPOUT,
                                 lora_target_modules=LORA_TARGET_MODULES)
model.to(device)

if RESUME:
    loaded = recorder.load_best_state(model)
    if not loaded:
        print(f"[{TAG}] không có best_state.pt trước đó -- bắt đầu LoRA từ đầu")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"tham số: {trainable/1e6:.1f}M trainable / {total/1e6:.0f}M tổng "
      f"({trainable/max(total,1):.2%})")

# Chạy thử 1 batch: model có cho ra điểm hữu hạn, và có phân biệt được các
# đoạn văn khác nhau không? (thay cho tc.choose_precision/check_pooling_discriminates
# vốn giả định model load fp32 + autocast -- không khớp base fp16 đóng băng + LoRA fp32 ở đây)
probe_docs = examples[0].negatives[:4]
ft.sanity_check(model, tokenizer, documents, probe_docs, device, MAX_LENGTH,
                INSTRUCTION, ft.SYSTEM_PROMPT, TAG)

if device == "cuda":
    free, total_mem = torch.cuda.mem_get_info()
    print(f"  GPU memory sau khi load: {(total_mem-free)/2**30:.2f} / {total_mem/2**30:.2f} GB")

model = tc.wrap_parallel(model, n_gpus)     # no-op nếu chỉ 1 GPU (Colab thường vậy)

## Cell 6 — Optimizer, scheduler, và hai hàm chạy

`train_one_epoch(n)` huấn luyện một epoch. `evaluate_and_record(n)` chấm lại
model ở **vai trò kênh `jina`** (thay `score_overrides={"jina": ...}`), ráp
vào LTR fusion 6 kênh, áp ngưỡng động α=0.15, rồi ghi xuống đĩa ngay.

In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    max_length=MAX_LENGTH, eval_batch_size=EVAL_BATCH_SIZE, batch_size=BATCH_SIZE,
    accum=ACCUM, negatives=NEGATIVES, passages_per_doc=PASSAGES_PER_DOC_TRAIN,
    warmup_ratio=.1, weight_decay=.01, max_grad_norm=1.0,
    instruction=INSTRUCTION, system_prompt=ft.SYSTEM_PROMPT, loss=LOSS, temperature=1.0,
)
tc.scale_for_gpus(args, n_gpus)     # no-op nếu n_gpus<=1, giữ nguyên effective batch nếu >1

sampler = tc.GroupSampler(examples, documents, NEGATIVES, PASSAGES_PER_DOC_TRAIN, seed=SEED)
groups_per_epoch = (len(examples) + args.batch_size - 1) // args.batch_size
total_steps = max(1, EPOCHS * groups_per_epoch // args.accum)

optimizer = tc.make_optimizer(tc.unwrap(model), LR, args.weight_decay)
scheduler = tc.make_scheduler(optimizer, total_steps, args.warmup_ratio)

print(f"{len(examples)} truy vấn, {groups_per_epoch} nhóm/epoch, "
      f"{total_steps} bước optimizer cho {EPOCHS} epoch")


def train_one_epoch(epoch):
    if recorder.already_done(epoch):
        print(f"epoch {epoch} đã ghi từ lần chạy trước — bỏ qua")
        return None
    started = time.perf_counter()
    loss = ft.train_epoch(model, tokenizer, sampler, args, optimizer, scheduler, device, epoch)
    print(f"epoch {epoch}: loss={loss:.4f} ({(time.perf_counter()-started)/60:.1f} phút)")
    return loss


def evaluate_and_record(epoch, loss=None):
    if recorder.already_done(epoch):
        print(f"epoch {epoch} đã ghi từ lần chạy trước — bỏ qua")
        return
    started = time.perf_counter()
    metrics, ranked, predictions, scores = ft.evaluate(model, tokenizer, bundle, args, device, TAG)
    improved = recorder.consider(epoch, metrics, ft.lora_state_dict(model), ranked, predictions,
                                 extra={"train_loss": loss})
    if improved:
        recorder.save_channel_scores(scores)
    recorder.package()          # zip nhỏ (không kèm .pt), tải về được ngay
    print(f"  kênh `jina` (thay bằng `{TAG}`): R@5={metrics['channel_only']['Recall@5']:.4f} "
          f"(cache gốc jina: {bc.channel_metrics(bundle, bundle.scores['jina'])['Recall@5']:.4f})")
    print(f"  chấm điểm mất {(time.perf_counter()-started)/60:.1f} phút")

## Cell 7 — Epoch 1

Kết quả tự ghi thẳng vào Drive, không cần tải gì.

In [ ]:
loss = train_one_epoch(1)
evaluate_and_record(1, loss)

## Cell 8 — Epoch 2

Kết quả tự ghi thẳng vào Drive, không cần tải gì.

In [ ]:
loss = train_one_epoch(2)
evaluate_and_record(2, loss)

## Cell 9 — Epoch 3

Epoch cuối (theo `EPOCHS` cấu hình ở Cell 2 — đổi `EPOCHS` và thêm/bớt cell
nếu muốn chạy khác 3 epoch).

In [ ]:
loss = train_one_epoch(3)
evaluate_and_record(3, loss)

## Cell 10 — Tổng kết

Bảng lịch sử đầy đủ và danh sách file để tải về.

In [ ]:
history = json.loads((recorder.dir / "history.json").read_text(encoding="utf-8"))

print(f"{'epoch':>9s} {'recall':>8s} {'prec':>7s} {'f2':>7s} {'kênh riêng':>11s}"
      f"  {'đã lưu':>7s}  vượt baseline")
for row in history["history"]:
    channel_only = row.get("channel_only", {}).get("Recall@5")
    print(f"{str(row['epoch']):>9s} {row['recall']:8.4f} {row['precision']:7.4f} "
          f"{row['f2']:7.4f} {(f'{channel_only:.4f}' if channel_only else '-'):>11s}  "
          f"{('BEST' if row.get('improved') else ''):>7s}  "
          f"{'YES' if row.get('beats_baseline') else ''}")

print(f"\nEpoch tốt nhất theo holdout 600 truy vấn: {history['best_epoch']}")
if history["best_epoch"] is None:
    print("Chưa epoch nào được chấm điểm — chưa có gì để lưu.")
elif not history.get("best_beats_baseline"):
    print("Trọng số epoch này ĐÃ được lưu vào best_state.pt (chỉ adapter LoRA, vài MB), "
          "nhưng vẫn dưới baseline cache -- bản nộp vẫn nên dùng điểm cache gốc.")
else:
    print("Epoch này vừa tốt nhất trong lần chạy, vừa vượt baseline cache -- "
          "ứng viên nghiêm túc để thay hẳn kênh `jina`.")
print("\nNgưỡng nhiễu của bài này là std 0.008 trên Recall. Chênh lệch nhỏ hơn "
      "khoảng đó không phải bằng chứng của gì cả -- dự án đã có 4 lần "
      "'thắng CV, thua leaderboard thật'.")

print(f"\nFile trong {WORK / TAG}:")
for path in sorted((WORK / TAG).rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(WORK)}  ({path.stat().st_size/2**20:.2f} MB)")